# تمرین ۲ — طبقه‌بندی تصویر: ViT در برابر ResNet-50 (دیتاست Animals-10، زیرمجموعه)

In [ ]:
# blocks output in Colab 💄
%%capture

! pip install datasets transformers

## ۱. بارگذاری دیتاست

دیتاست `Rapidata/Animals-10` (HuggingFace Datasets): ۱۰ کلاس حیوان (Butterfly, Cat, Chicken, Cow, Dog, Elephant, Horse, Sheep, Spider, Squirrel)، در اصل ۲۳٬۵۵۴ تصویر — برای سرعت، یک زیرمجموعه‌ی تصادفی 1200 تایی برمی‌داریم (حداقل ۱۰۰۰ طبق خواسته).

In [ ]:
from datasets import load_dataset, DatasetDict

full_ds = load_dataset("Rapidata/Animals-10")["train"]  # 23,554 images, 10 classes

# زیرمجموعه‌ی تصادفی برای سرعت بیشتر در آموزش (حداقل ۱۰۰۰ تصویر)
subset = full_ds.shuffle(seed=42).select(range(1200))

split = subset.train_test_split(test_size=0.2, seed=42)
ds = DatasetDict({"train": split["train"], "validation": split["test"]})
ds

نمونه‌ی یک داده

In [ ]:
ex = ds['train'][0]
ex

نمایش تصویر نمونه

In [ ]:
image = ex['image']
image

برچسب کلاس نمونه

In [ ]:
labels = ds['train'].features['label']

نام کلاس برچسب

In [ ]:
labels.int2str(ex['label'])

نمایش چند نمونه از هر کلاس حیوان

In [ ]:
import random
from PIL import ImageDraw, ImageFont, Image

def show_examples(ds, seed: int = 1234, examples_per_class: int = 3, size=(350, 350)):
    w, h = size
    labels = ds['train'].features['label'].names
    grid = Image.new('RGB', size=(examples_per_class * w, len(labels) * h))
    draw = ImageDraw.Draw(grid)
    font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationMono-Bold.ttf", 24)

    for label_id, label in enumerate(labels):
        pool = ds['train'].filter(lambda ex: ex['label'] == label_id)
        n = min(examples_per_class, pool.num_rows)
        if n == 0:
            continue
        ds_slice = pool.shuffle(seed).select(range(n))
        for i, example in enumerate(ds_slice):
            image = example['image']
            idx = examples_per_class * label_id + i
            box = (idx % examples_per_class * w, idx // examples_per_class * h)
            grid.paste(image.resize(size), box=box)
            draw.text(box, label, (255, 255, 255), font=font)
    return grid

show_examples(ds, seed=random.randint(0, 1337), examples_per_class=3)

بررسی چشمی نمونه‌ها

## ۲. پیش‌پردازش تصویر با Feature/Image Processor مدل ViT

In [ ]:
from transformers import AutoImageProcessor

model_name_or_path = 'google/vit-base-patch16-224-in21k'
feature_extractor = AutoImageProcessor.from_pretrained(model_name_or_path)

تنظیمات Feature Extractor

In [ ]:
feature_extractor

خروجی پردازش یک تصویر (pixel_values)

In [ ]:
feature_extractor(image, return_tensors='pt')

## ۳. تابع پردازش کل دیتاست

In [ ]:
def process_example(example):
    inputs = feature_extractor(example['image'], return_tensors='pt')
    inputs['labels'] = example['label']
    return inputs

In [ ]:
process_example(ds['train'][0])

اعمال پردازش روی کل دیتاست با `with_transform`

In [ ]:
def transform(example_batch):
    # Take a list of PIL images and turn them to pixel values
    inputs = feature_extractor([x for x in example_batch['image']], return_tensors='pt')
    # Don't forget to include the labels!
    inputs['labels'] = example_batch['label']
    return inputs

اعمال transform روی دیتاست

In [ ]:
prepared_ds = ds.with_transform(transform)

Now, whenever we get an example from the dataset, our transform will be
applied in real time (on both samples and slices, as shown below)

In [ ]:
prepared_ds['train'][0:2]

## ۴. آموزش و ارزیابی مدل اول (ViT)

### Data Collator

In [ ]:
import torch

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

### معیار ارزیابی: Accuracy

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")
def compute_metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

### بارگذاری مدل ViT

In [ ]:
from transformers import ViTForImageClassification

labels = ds['train'].features['label'].names

model = ViTForImageClassification.from_pretrained(
    model_name_or_path,
    num_labels=len(labels),
    id2label={str(i): c for i, c in enumerate(labels)},
    label2id={c: str(i) for i, c in enumerate(labels)}
)

### تنظیمات آموزش (TrainingArguments)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir="./vit-base-animals10-demo",
  per_device_train_batch_size=16,
  eval_strategy="steps",
  num_train_epochs=4,
  fp16=True,
  save_steps=100,
  eval_steps=100,
  logging_steps=10,
  learning_rate=2e-4,
  seed=42,
  save_total_limit=2,
  remove_unused_columns=False,
  push_to_hub=False,
  report_to='tensorboard',
  load_best_model_at_end=True,
)

### تعریف Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=prepared_ds["train"],
    eval_dataset=prepared_ds["validation"],
    tokenizer=feature_extractor,
)

In [ ]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

In [ ]:
metrics = trainer.evaluate(prepared_ds['validation'])
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

In [ ]:
kwargs = {
    "finetuned_from": model.config._name_or_path,
    "tasks": "image-classification",
    "dataset": 'Rapidata/Animals-10',
    "tags": ['image-classification'],
}

if training_args.push_to_hub:
    trainer.push_to_hub('done', **kwargs)
else:
    trainer.create_model_card(**kwargs)

## ۵. مدل دوم: ResNet-50
همان مراحل بالا، این‌بار با `microsoft/resnet-50` و **دقیقاً همان تنظیمات آموزش** (برای مقایسه‌ی منصفانه).

### پردازش تصویر برای ResNet-50 (استفاده از همان `ds`)

In [ ]:
from transformers import AutoImageProcessor

model_name_or_path_2 = "microsoft/resnet-50"
image_processor_2 = AutoImageProcessor.from_pretrained(model_name_or_path_2)

def transform_resnet(example_batch):
    inputs = image_processor_2([x for x in example_batch['image']], return_tensors='pt')
    inputs['labels'] = example_batch['label']
    return inputs

prepared_ds_2 = ds.with_transform(transform_resnet)

### بارگذاری مدل ResNet-50

In [ ]:
from transformers import AutoModelForImageClassification

labels_2 = ds['train'].features['label'].names

model_2 = AutoModelForImageClassification.from_pretrained(
    model_name_or_path_2,
    num_labels=len(labels_2),
    id2label={str(i): c for i, c in enumerate(labels_2)},
    label2id={c: str(i) for i, c in enumerate(labels_2)},
    ignore_mismatched_sizes=True,
)

### تنظیمات آموزش (دقیقاً مثل ViT: batch=16, epoch=4, lr=2e-4)

In [ ]:
training_args_2 = TrainingArguments(
  output_dir="./resnet50-animals10-demo",
  per_device_train_batch_size=16,
  eval_strategy="steps",
  num_train_epochs=4,
  fp16=True,
  save_steps=100,
  eval_steps=100,
  logging_steps=10,
  learning_rate=2e-4,
  seed=42,
  save_total_limit=2,
  remove_unused_columns=False,
  push_to_hub=False,
  report_to='tensorboard',
  load_best_model_at_end=True,
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    train_dataset=prepared_ds_2["train"],
    eval_dataset=prepared_ds_2["validation"],
    tokenizer=image_processor_2,
)

### آموزش مدل دوم

In [ ]:
train_results_2 = trainer_2.train()
trainer_2.save_model()
trainer_2.log_metrics("train", train_results_2.metrics)
trainer_2.save_metrics("train", train_results_2.metrics)
trainer_2.save_state()

### ارزیابی مدل دوم

In [ ]:
metrics_2 = trainer_2.evaluate(prepared_ds_2['validation'])
trainer_2.log_metrics("eval", metrics_2)
trainer_2.save_metrics("eval", metrics_2)

## ۶. جدول مقایسه نتایج (بخش ۲.۵ گزارش)

In [ ]:
import pandas as pd

results_table = pd.DataFrame([
    {
        "دیتاست": "Rapidata/Animals-10 (subset)",
        "مدل": "ViT (google/vit-base-patch16-224-in21k)",
        "Accuracy": metrics.get("eval_accuracy"),
        "زمان آموزش (ثانیه)": train_results.metrics.get("train_runtime"),
        "تعداد Epoch": training_args.num_train_epochs,
    },
    {
        "دیتاست": "Rapidata/Animals-10 (subset)",
        "مدل": "ResNet-50 (microsoft/resnet-50)",
        "Accuracy": metrics_2.get("eval_accuracy"),
        "زمان آموزش (ثانیه)": train_results_2.metrics.get("train_runtime"),
        "تعداد Epoch": training_args_2.num_train_epochs,
    },
])
results_table

**نتایج واقعی اجرا (برای رجوع، اگر دوباره اجرا نکنی همین‌ها معتبرن):**

| دیتاست | مدل | Accuracy | زمان آموزش (ثانیه) | Epoch |
|---|---|---|---|---|
| Rapidata/Animals-10 (subset) | ViT (google/vit-base-patch16-224-in21k) | 0.9875 | 76.61 | 4 |
| Rapidata/Animals-10 (subset) | ResNet-50 (microsoft/resnet-50) | 0.9250 | 34.16 | 4 |


## ۷. پاسخ سوالات تحلیلی (۲.۶)

**۱. کدام مدل دقت بالاتری داشت؟ چرا؟ آیا با انتظار (شهرت مدل‌ها) همخوانی داشت؟**
ViT با Accuracy=۰.۹۸۷۵ به‌وضوح از ResNet-50 (Accuracy=۰.۹۲۵۰) بهتر بود. دلیل اصلی، پیش‌آموزش قوی ViT روی ImageNet-21k (دیتاست بسیار بزرگ‌تر از ImageNet-1k که ResNet-50 روش پیش‌آموزش دیده) و مکانیزم self-attention آن است که وابستگی‌های سراسری تصویر را بهتر مدل می‌کند. این نتیجه کمی برخلاف تصور رایج است، چون معمولاً گفته می‌شود ترنسفورمرها (مثل ViT) به داده‌ی بیشتری نسبت به CNN نیاز دارند تا برتری نشان دهند؛ اینجا با وجود دیتاست کوچک (۱۲۰۰ تصویر)، پیش‌آموزش قوی‌تر ViT همچنان برتری داد.

**۲. تفاوت اصلی معماری ViT (ترنسفورمر) و ResNet (CNN) در پردازش تصویر چیست؟**
ResNet با لایه‌های کانولوشنی و ساختار محلی (local receptive field) روی الگوهای نزدیک به هم در تصویر تمرکز می‌کند و به‌تدریج با عمیق‌تر شدن شبکه ویژگی‌های سطح بالاتر را می‌سازد. ViT تصویر را به تکه‌های کوچک (patch) تقسیم و با self-attention رابطه‌ی هر تکه را با تمام تکه‌های دیگر می‌سنجد، یعنی از همان ابتدا دید سراسری به تصویر دارد؛ در عوض، فرضیات ساختاری کمتری (inductive bias کمتر) نسبت به CNN دارد و معمولاً به داده‌ی پیش‌آموزش بیشتری نیاز دارد.

**۳. آیا افزایش epoch باعث بهبود محسوس دقت شد یا مدل به overfitting نزدیک شد؟**
برای هر دو مدل، در طول ۴ epoch، Accuracy پیوسته افزایش و Validation Loss پیوسته کاهش یافت (ViT: از ۰.۷۲ به ۰.۹۸۷۵؛ ResNet-50: از ۰.۷۲ به ۰.۹۲۵)، یعنی نشانه‌ای از overfitting دیده نشد. با این‌حال روند بهبود ResNet-50 بین epoch های آخر (step۲۰۰ به step۲۴۰) کند‌تر شد (۰.۹۰۸ به ۰.۹۲۵)، که می‌تواند نشانه‌ی نزدیک شدن به یک سقف با همین حجم داده باشد؛ افزایش epoch بیشتر برای بررسی دقیق‌تر overfitting توصیه می‌شود.

**۴. آیا کلاس‌ها متوازن‌اند؟ اثرش روی نتایج چه بود؟**
دیتاست اصلی Animals-10 نامتوازن است (مثلاً کلاس‌هایی مثل sheep و elephant تصاویر کمتری نسبت به dog یا spider دارند)؛ چون زیرمجموعه‌ی ۱۲۰۰ تایی به‌صورت تصادفی از کل دیتاست گرفته شده، همان نامتوازنی نسبی در آن هم وجود دارد. این یعنی Accuracy کلی می‌تواند به‌خاطر عملکرد خوب روی کلاس‌های پرتعداد بالا به‌نظر برسد، در حالی‌که عملکرد روی کلاس‌های کم‌تعداد ممکن است ضعیف‌تر باشد؛ برای بررسی دقیق این موضوع باید علاوه‌بر Accuracy کلی، معیارهایی مثل per-class precision/recall یا F1 هم محاسبه شود که در این اجرا محاسبه نشده است.

مدل اول (ViT) آموزش و ارزیابی شد. ادامه: مدل دوم (ResNet-50).